In [ ]:
# --- STEP 1: IMPORT LIBRARIES ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import statsmodels.api as sm
from scipy.stats import pearsonr
import plotly.graph_objects as go
import altair as alt
import json
import requests
!pip install pgeocode
import pgeocode

In [ ]:
# --- STEP 2: LOAD DATA ---
df_insp_raw = pd.read_csv('2024_Food_Establishment_Inspection_Scores_Edited.csv')
df_census_raw = pd.read_csv('Census_Data.csv')

In [ ]:
# --- STEP 3: CLEANING & PREPROCESSING ---

# 3.1 Clean Inspection Data
df_insp = df_insp_raw.dropna(subset=['Zip Code', 'Score']).copy()
df_insp['Zip Code'] = df_insp['Zip Code'].astype(str).str[:5]
df_insp['Score'] = pd.to_numeric(df_insp['Score'], errors='coerce')
df_insp['Inspection Date'] = pd.to_datetime(df_insp['Inspection Date'], errors='coerce')
df_insp['Facility ID'] = df_insp['Facility ID'].astype(str)
df_insp['Restaurant Type'] = df_insp['Restaurant Type'].str.strip().str.capitalize()
df_insp = df_insp[(df_insp['Score'] >= 0) & (df_insp['Score'] <= 100)]

In [ ]:
# --- INSPECTION SCORE SANITY CHECKS ---
df_insp
print("shape: ", df_insp.shape)
print("head: ", df_insp.head(5))
print("tail: ", df_insp.tail(5))
print("info: ", df_insp.info())
print("describe: ", df_insp.describe())

shape:  (20761, 8)
head:             Restaurant Name Zip Code Inspection Date  Score  \
0   Boulevards Grill & Bar    78731      2026-03-11   88.0   
1   Los Pepes Mexican Food    78723      2026-03-11   78.0   
2  MN - Chokdee Restaurant    78653      2026-03-11   87.0   
3   Speedy Stop Store #219    78723      2026-03-11   92.0   
4   Kentucky Fried Chicken    78723      2026-03-11   96.0   

                             Address Facility ID Process Description  \
0          3616 Far West Blvd Austin     2802419  Routine Inspection   
1      4700 Loyola Ln Ste 115 Austin    10818868  Routine Inspection   
2        11300 E US 290 Hwy WB Manor    11647869  Routine Inspection   
3              1660 E 51st St Austin     2803642  Routine Inspection   
4  7206  ED BLUESTEIN SB BLVD AUSTIN    11789720  Routine Inspection   

  Restaurant Type  
0             NaN  
1             NaN  
2             NaN  
3             NaN  
4             NaN  
tail:                     Restaurant Name Zip Co

In [ ]:
# 3.2 Clean Census Data
df_census = df_census_raw.iloc[1:].copy()
# Extract 5-digit zip and clean Median Income (S1903_C03_001E)
df_census['Zip Code'] = df_census['NAME'].str.replace('ZCTA5 ', '', regex=False).str.strip().str[:5]
df_census['Median_Income'] = (
    df_census['S1903_C03_001E']
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
    .str.replace('-', '', regex=False)
)

df_census['Median_Income'] = pd.to_numeric(df_census['Median_Income'], errors='coerce')
df_census_final = df_census[['Zip Code', 'Median_Income']].dropna()

In [ ]:
# --- CENSUS DATA SANITY CHECKS ---
df_census_final
print("shape: ", df_census_final.shape)
print("head: ", df_census_final.head(5))
print("tail: ", df_census_final.tail(5))
print("info: ", df_census_final.info())
print("describe: ", df_census_final.describe())

shape:  (61, 2)
head:    Zip Code  Median_Income
1    76574        78568.0
2    78610       119698.0
3    78612        95496.0
4    78613       125487.0
5    78615        80938.0
tail:     Zip Code  Median_Income
59    78754        90121.0
60    78756        94701.0
61    78757       103860.0
62    78758        74381.0
63    78759       102400.0
<class 'pandas.core.frame.DataFrame'>
Index: 61 entries, 1 to 63
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Zip Code       61 non-null     object 
 1   Median_Income  61 non-null     float64
dtypes: float64(1), object(1)
memory usage: 1.4+ KB
info:  None
describe:         Median_Income
count      61.000000
mean   110658.032787
std     39696.219024
min     32380.000000
25%     84573.000000
50%    102400.000000
75%    128000.000000
max    238559.000000


In [ ]:
# --- 2024 Inspection Data ---
df_2024 = df_insp[df_insp['Inspection Date'].dt.year == 2024].copy()
df_2024

,Restaurant Name,Zip Code,Inspection Date,Score,Address,Facility ID,Process Description,Restaurant Type
8773,Berkman Food Mart,78723,2024-12-31,92.0,6615 Berkman Dr Austin,2803871,Routine Inspection,Local
8774,El Pollo Regio,78723,2024-12-31,89.0,6615 Berkman Dr Bldg B Austin,12393520,Routine Inspection,Chain
8775,Prime Mart,78753,2024-12-31,93.0,12430 N Lamar Blvd Austin,10003038,Routine Inspection,Local
8776,Pita Fusion,78757,2024-12-31,97.0,2525 W Anderson Ln Unit 200 Austin,12397876,Routine Inspection,Chain
8777,Tropicana Cuban Resturant,78753,2024-12-31,93.0,9616 N Lamar Blvd Ste 141 Austin,12397576,Routine Inspection,Local
...,...,...,...,...,...,...,...,...
15547,ZZa Pizza + Salad,78738,2024-01-02,92.0,15511 W SH 71 UNIT 150 BEE CAVE,12186387,Routine Inspection,Local
15548,Caffe Medici,78705,2024-01-02,89.0,2222 Guadalupe St Austin,2800910,Routine Inspection,Chain
15549,Tokyo Garden Catering,78731,2024-01-02,98.0,7025 Village Center Dr Austin,2802634,Routine Inspection,Chain
15550,"Snooze, an A.M. Eatery",78756,2024-01-02,97.0,3800 N Lamar Blvd Ste 120 Austin,11520599,Routine Inspection,Chain


In [ ]:
# --- STEP 4: MERGE AND RESET INDEX ---
df_2024_final = pd.merge(df_2024, df_census_final, on='Zip Code', how='inner')
df_2024_final = df_2024_final.reset_index(drop=True)

In [ ]:
# --- FINAL DATASET SANITY CHECKS ---
df_2024_final

,Restaurant Name,Zip Code,Inspection Date,Score,Address,Facility ID,Process Description,Restaurant Type,Median_Income
0,Berkman Food Mart,78723,2024-12-31,92.0,6615 Berkman Dr Austin,2803871,Routine Inspection,Local,100329.0
1,El Pollo Regio,78723,2024-12-31,89.0,6615 Berkman Dr Bldg B Austin,12393520,Routine Inspection,Chain,100329.0
2,Prime Mart,78753,2024-12-31,93.0,12430 N Lamar Blvd Austin,10003038,Routine Inspection,Local,65170.0
3,Pita Fusion,78757,2024-12-31,97.0,2525 W Anderson Ln Unit 200 Austin,12397876,Routine Inspection,Chain,103860.0
4,Tropicana Cuban Resturant,78753,2024-12-31,93.0,9616 N Lamar Blvd Ste 141 Austin,12397576,Routine Inspection,Local,65170.0
...,...,...,...,...,...,...,...,...,...
6677,ZZa Pizza + Salad,78738,2024-01-02,92.0,15511 W SH 71 UNIT 150 BEE CAVE,12186387,Routine Inspection,Local,183048.0
6678,Caffe Medici,78705,2024-01-02,89.0,2222 Guadalupe St Austin,2800910,Routine Inspection,Chain,32380.0
6679,Tokyo Garden Catering,78731,2024-01-02,98.0,7025 Village Center Dr Austin,2802634,Routine Inspection,Chain,105494.0
6680,"Snooze, an A.M. Eatery",78756,2024-01-02,97.0,3800 N Lamar Blvd Ste 120 Austin,11520599,Routine Inspection,Chain,94701.0


Argument 1 - Looking at the relationship between inspection scores and median income within zip codes

In [ ]:
df_merged = pd.merge(df_2024, df_census_final, on='Zip Code', how='inner')

zip_stats = df_merged.groupby('Zip Code').agg({
    'Score': 'mean',
    'Median_Income': 'first',
    'Restaurant Name': 'count'
}).reset_index()

zip_stats.columns = ['Zip Code', 'Avg_Score', 'Median_Income', 'Inspection_Frequency']
zip_stats['Avg_Score'] = zip_stats['Avg_Score'].round(2)

# --- VISUAL 1: Median Income vs. Inspection Score ---
fig1 = px.scatter(
    zip_stats,
    x='Median_Income',
    y='Avg_Score',
    size='Inspection_Frequency', # Bubble size represents frequency
    color='Avg_Score',
    hover_name='Zip Code',
    hover_data={
        'Median_Income': ':$,.0f', # Format as currency
        'Avg_Score': ':.2f',
        'Inspection_Frequency': True
    },
    trendline="ols",
    title='Median Income vs. Average Inspection Score (2024)',
    labels={'Median_Income': 'Median Income ($)', 'Avg_Score': 'Avg Inspection Score'}
)
fig1.update_layout(xaxis_tickformat='$,.0f')

# To include the trend line equation alongside the $R^2$ value, you can extract the coefficients (the slope and the intercept) from the same model results object used for the $R^2$.Here is the updated code block that extracts both and formats them into a single label:Python# 1. Get the trendline results
results = px.get_trendline_results(fig1)
model_results = results.px_fit_results.iloc[0]

# 2. Extract statistics
r_squared = model_results.rsquared
intercept, slope = model_results.params  # params[0] is intercept, params[1] is slope

# 3. Format the equation string
# Using scientific notation for slope if it's very small (common with large x-axis values like Income)
equation = f"y = {slope:.4e}x + {intercept:.2f}"
r2_text = f"R² = {r_squared:.3f}"

# 4. Add the combined label as an annotation
fig1.add_annotation(
    x=0.05, y=0.05,
    xref="paper", yref="paper",
    text=f"<b>Regression Analysis:</b><br>{equation}<br>{r2_text}",
    showarrow=False,
    align="left",
    font=dict(size=12, color="black"),
    bgcolor="rgba(255, 255, 255, 0.8)",
    bordercolor="black",
    borderwidth=1
)

fig1.add_annotation(
    x=32380, y=89.63,             # The actual data coordinates
    xref="x", yref="y",           # Anchor to the data axes
    text="UT",       # The label text
    showarrow=True,
    arrowhead=2,
    ax=20, ay=-50,
    font=dict(size=12, color="blue"),
    bgcolor="white",
    opacity=0.8
)

fig1.show()

In [ ]:
df_merged

,Restaurant Name,Zip Code,Inspection Date,Score,Address,Facility ID,Process Description,Restaurant Type,Median_Income
0,Berkman Food Mart,78723,2024-12-31,92.0,6615 Berkman Dr Austin,2803871,Routine Inspection,Local,100329.0
1,El Pollo Regio,78723,2024-12-31,89.0,6615 Berkman Dr Bldg B Austin,12393520,Routine Inspection,Chain,100329.0
2,Prime Mart,78753,2024-12-31,93.0,12430 N Lamar Blvd Austin,10003038,Routine Inspection,Local,65170.0
3,Pita Fusion,78757,2024-12-31,97.0,2525 W Anderson Ln Unit 200 Austin,12397876,Routine Inspection,Chain,103860.0
4,Tropicana Cuban Resturant,78753,2024-12-31,93.0,9616 N Lamar Blvd Ste 141 Austin,12397576,Routine Inspection,Local,65170.0
...,...,...,...,...,...,...,...,...,...
6677,ZZa Pizza + Salad,78738,2024-01-02,92.0,15511 W SH 71 UNIT 150 BEE CAVE,12186387,Routine Inspection,Local,183048.0
6678,Caffe Medici,78705,2024-01-02,89.0,2222 Guadalupe St Austin,2800910,Routine Inspection,Chain,32380.0
6679,Tokyo Garden Catering,78731,2024-01-02,98.0,7025 Village Center Dr Austin,2802634,Routine Inspection,Chain,105494.0
6680,"Snooze, an A.M. Eatery",78756,2024-01-02,97.0,3800 N Lamar Blvd Ste 120 Austin,11520599,Routine Inspection,Chain,94701.0


Argument 1 Conclusion: No relationship between average restaurant inspection scores and median income suggests that socioeconomic status does not dictate food safety compliance, indicating that inspection systems are likely applying uniform standards regardless of neighborhood wealth. It implies that high-income areas are not inherently cleaner, and low-income areas are not inherently less hygienic

Argument 2 - Looking at relationship between inspection frequency and average inspection score within zip codes.

In [ ]:
# 1. Prepare Census Data
df_census = df_census_raw.iloc[1:].copy()
df_census['Zip Code'] = df_census['NAME'].str.replace('ZCTA5 ', '', regex=False).str.strip().str[:5]

# Convert columns to numeric
df_census['Household_Count'] = pd.to_numeric(df_census['S1903_C01_001E'], errors='coerce')
# Assuming df_2024 already has Median_Income from your previous steps
df_census['Median_Income'] = pd.to_numeric(df_census['S1903_C03_001E'], errors='coerce')

# 2. Merge with your inspection data (df_2024)
# Make sure df_2024 has 'Zip Code', 'Score', and 'Restaurant Name'
df_merged = pd.merge(df_2024, df_census[['Zip Code', 'Household_Count', 'Median_Income']], on='Zip Code', how='inner')

# 3. Aggregate by Zip Code
fig2_stats = df_merged.groupby('Zip Code').agg({
    'Score': 'mean',
    'Median_Income': 'first',
    'Restaurant Name': 'count',
    'Household_Count': 'first'
}).reset_index()

# Rename columns to match what you use in the scatter plot
fig2_stats.columns = ['Zip Code', 'Avg_Score', 'Median_Income', 'Inspection_Frequency', 'Household_Count']

# 4. Create the Visualization
fig2 = px.scatter(
    fig2_stats,
    x='Inspection_Frequency',
    y='Avg_Score',
    size='Household_Count',      # Bubble size represents household volume
    color='Avg_Score',
    hover_name='Zip Code',
    hover_data={
        'Inspection_Frequency': True,
        'Avg_Score': ':.2f',
        'Median_Income': ':$,.0f',
        'Household_Count': ':,'  # Shows commas in large numbers
    },
    trendline="ols",
    title='Restaurant Inspection Frequency vs. Average Score (Sized by Households)',
    labels={
        'Inspection_Frequency': 'Total Number of Inspections',
        'Avg_Score': 'Avg Inspection Score',
        'Household_Count': 'Total Households'
    }
)

# --- 5. EXTRACT TRENDLINE RESULTS ---
results = px.get_trendline_results(fig2)
model_results = results.px_fit_results.iloc[0]

# Get the stats
r_squared = model_results.rsquared
intercept, slope = model_results.params
equation = f"y = {slope:.4f}x + {intercept:.2f}"

# --- 6. ADD ANNOTATIONS ---

# Add the Regression Analysis box to the bottom-left
fig2.add_annotation(
    x=0.90, y=0.05,               # Bottom-left of the chart area
    xref="paper", yref="paper",   # Use 0-1 coordinate system
    text=f"<b>Regression Analysis:</b><br>{equation}<br>R² = {r_squared:.3f}",
    showarrow=False,
    align="left",
    font=dict(size=12, color="black"),
    bgcolor="rgba(255, 255, 255, 0.8)",
    bordercolor="black",
    borderwidth=1
)

point_78705 = fig2_stats[fig2_stats['Zip Code'] == '78705'].iloc[0]

fig2.add_annotation(
    x=point_78705['Inspection_Frequency'],
    y=point_78705['Avg_Score'],
    text="UT",
    showarrow=True,
    arrowhead=2,
    xref="x", yref="y",
    ax=0, ay=-40,    # 0 centers it, -40 moves it ABOVE the dot
    font=dict(color="blue", size=11)
)

fig2.show()

Argument 2 Conclusion:

Argument 3 - Looking at how inspection scores change over the years


In [ ]:
# --- 1. DATA PREPARATION (Creating df_time) ---
df = pd.read_csv('2024_Food_Establishment_Inspection_Scores_Edited.csv')
df['Inspection Date'] = pd.to_datetime(df['Inspection Date'], errors='coerce')
df = df.dropna(subset=['Inspection Date', 'Score'])

# Standardize Zip Codes and Extract Year
df['Zip Code'] = df['Zip Code'].astype(str).str[:5]
df['Inspection Year'] = df['Inspection Date'].dt.year

# Aggregate scores by Zip and Year
df_time = df.groupby(['Zip Code', 'Inspection Year']).agg({
    'Score': 'mean'
}).reset_index()
df_time.rename(columns={'Score': 'Average Inspection Score'}, inplace=True)

# Important Zip Codes
important_zips = ['78701', '78705', '78739', '78704', '78621', '78641', '78751', '78721']

fig3 = go.Figure()

for zip_code in df_time["Zip Code"].unique():
    # Filter and sort by year to ensure lines connect chronologically
    temp = df_time[df_time["Zip Code"] == zip_code].sort_values("Inspection Year")

    is_important = str(zip_code) in important_zips

    fig3.add_trace(go.Scatter(
        x=temp["Inspection Year"],
        y=temp["Average Inspection Score"],
        mode="lines+markers",
        name=str(zip_code),

        visible=True if is_important else "legendonly",  # Only important ones show at start
        line=dict(width=4 if is_important else 1.5),     # "Bold" the line width for important ones
        opacity=1.0 if is_important else 0.4,            # Slightly fade the background ones

        hovertemplate=(
            "<b>ZIP Code: " + str(zip_code) + "</b><br>" +
            "Inspection Year: %{x}<br>" +
            "Average Score: %{y:.2f}<extra></extra>"
        )
    ))

fig3.update_layout(
    title="Average Restaurant Inspection Scores Over Time by ZIP Code",
    xaxis_title="Inspection Year",
    yaxis_title="Average Inspection Score",
    template="plotly_white",
    width=1000,
    height=600,
    yaxis=dict(range=[75, 101]),
    xaxis=dict(dtick=1),
    legend_title="ZIP Codes (Click to toggle)"
)

fig3.show()

In [ ]:
df = df.dropna(subset=["Zip Code", "Score"]).copy()

# Keep Austin-area ZIPs: 786 + 787
df = df[df["Zip Code"].str.match(r"^(786|787)")].copy()

# --- STEP 2: AGGREGATE BY ZIP ---
df_census = (
    df.groupby("Zip Code", as_index=False)
      .agg(
          Average_Inspection_Score=("Score", "mean"),
          Inspection_Count=("Score", "count")
      )
)

# --- STEP 3: GET ZIP CENTROID COORDINATES ---
nomi = pgeocode.Nominatim("us")

df_census["Latitude"] = df_census["Zip Code"].apply(
    lambda z: nomi.query_postal_code(z).latitude
)
df_census["Longitude"] = df_census["Zip Code"].apply(
    lambda z: nomi.query_postal_code(z).longitude
)

# Drop rows where coordinates were not found
df_census = df_census.dropna(subset=["Latitude", "Longitude"]).copy()
fig3_map = px.scatter_mapbox(
    df_census,
    lat="Latitude",
    lon="Longitude",
    color="Average_Inspection_Score",
    size="Inspection_Count",
    hover_name="Zip Code",
    hover_data={
        "Average_Inspection_Score": ":.2f",
        "Inspection_Count": True,
        "Latitude": False,
        "Longitude": False
    },
    color_continuous_scale="Plasma",
    zoom=9.5,
    center={"lat": 30.2672, "lon": -97.7431},
    title="Average Restaurant Inspection Scores by ZIP Code",
    size_max=30
)

fig3_map.update_layout(
    mapbox_style="carto-positron",
    template="plotly_white",
    margin={"r": 20, "t": 60, "l": 20, "b": 20}
)

fig3_map.show()

Argument 4 - Looking at the relationship between median household income and race.

In [ ]:
# Sort by income for better visualization
zip_income_sorted = zip_stats.sort_values('Median_Income', ascending=False)

fig_income = px.bar(
    zip_income_sorted,
    x='Zip Code',
    y='Median_Income',
    color='Median_Income',
    title='Median Income Distribution by Austin Zip Code',
    color_continuous_scale='Viridis',
    labels={'Median_Income': 'Median Household Income'}
)

fig_income.update_layout(xaxis={'categoryorder':'total descending'}, xaxis_tickangle=-45)
fig_income.show()

# 1. Aggregate scores by Zip AND Type
type_income_stats = df_2024_final.groupby(['Zip Code', 'Restaurant Type']).agg({
    'Score': 'mean',
    'Median_Income': 'first'
}).reset_index()

df_census = df_census_raw.iloc[1:].copy()
df_census['Zip Code'] = df_census['NAME'].str.replace('ZCTA5 ', '', regex=False).str.strip().str[:5]

df_census['Median_Income'] = (
    df_census['S1903_C03_001E']
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
    .str.replace('-', '', regex=False)
)
df_census['Median_Income'] = pd.to_numeric(df_census['Median_Income'], errors='coerce')
df_census_final = df_census[['Zip Code', 'Median_Income']].dropna()

df_final = pd.merge(df_2024, df_census_final, on='Zip Code', how='inner')

# 2. MAP: Geographic Distribution of Median Income
# Using a public TX Zip Code GeoJSON
repo_url = 'https://raw.githubusercontent.com/OpenDataDE/State-zip-code-GeoJSON/master/tx_texas_zip_codes_geo.min.json'

fig_map = px.choropleth(
    df_census_final,
    geojson=repo_url,
    locations='Zip Code',
    featureidkey="properties.ZCTA5CE10",
    color='Median_Income',
    color_continuous_scale="Viridis",
    labels={'Median_Income':'Median Household Income'},
    title='Austin Neighborhood Wealth Distribution'
)
fig_map.update_geos(fitbounds="locations", visible=False)
fig_map.show()

In [ ]:
# Define race columns
race_cols = {
    "S1903_C03_001E": "Overall",
    "S1903_C03_002E": "White",
    "S1903_C03_003E": "Black",
    "S1903_C03_005E": "Asian",
    "S1903_C03_009E": "Hispanic"
}

# Clean values
def clean_income(series):
    return pd.to_numeric(series.astype(str).str.replace(r"[,\+\-]", "", regex=True), errors="coerce")

for col in race_cols.keys():
    df_census[col] = clean_income(df_census[col])

# 🔥 KEY: KEEP ZIP LEVEL (NO AVERAGING)
df_plot = df_census[["Zip Code"] + list(race_cols.keys())].melt(
    id_vars="Zip Code",
    value_vars=list(race_cols.keys()),
    var_name="Group",
    value_name="Median Household Income"
)

# Map column codes → clean group names
df_plot["Group"] = df_plot["Group"].map(race_cols)

# Drop missing values
df_plot = df_plot.dropna(subset=["Zip Code", "Median Household Income"])

# Sort ZIPs
zip_order = sorted(df_plot["Zip Code"].unique(), key=lambda z: int(z))

# --- PLOT (THIS MATCHES YOUR IMAGE) ---
fig = px.scatter(
    df_plot,
    x="Zip Code",
    y="Median Household Income",
    color="Group",
    category_orders={"Zip Code": zip_order},
    title="Median Household Income by Race/Ethnicity Within ZIP Codes",
    labels={
        "Zip Code": "ZIP Code",
        "Median Household Income": "Median Household Income ($)",
        "Group": "Group"
    },
    hover_data={"Median Household Income": ":,.0f"}
)

fig.update_traces(marker=dict(size=9))

fig.update_layout(
    template="plotly_white",
    width=1700,
    height=600,
    title=dict(x=0.06),
    legend=dict(title="Group"),
    margin=dict(l=80, r=120, t=70, b=80)
)

fig.update_yaxes(range=[0, 200000])

fig.show()

Argument 5 - Looking at how local and chain restaurants are distributed across zip codes in Austin

In [ ]:
type_counts = df_2024_final.groupby(['Zip Code', 'Restaurant Type']).size().reset_index(name='Count')
zip_totals = df_2024_final.groupby('Zip Code').size().reset_index(name='Total_in_Zip')

# 3. Merge to calculate the relative composition percentage for EACH type
plot_df = pd.merge(type_counts, zip_totals, on='Zip Code')
plot_df['Composition_Pct'] = (plot_df['Count'] / plot_df['Total_in_Zip']) * 100

fig5 = px.bar(
    plot_df,
    x='Zip Code',
    y='Count',
    color='Restaurant Type',
    title='Restaurant Composition by Zip Code (2024)',
    hover_data={
        'Zip Code': True,
        'Restaurant Type': True,
        'Count': True,
        'Composition_Pct': ':.2f%',
        'Total_in_Zip': True
    },
    labels={
        'Count': 'Number of Establishments',
        'Composition_Pct': 'Composition %',
        'Total_in_Zip': 'Total Restaurants in this Zip'
    },
    barmode='stack',
    color_discrete_map={'Local': '#1B4332', 'Chain': '#00B4D8'}
)

fig5.update_layout(xaxis={'categoryorder':'total descending'})

fig5.show()

In [ ]:
# 1. Prepare base counts
type_counts = df_2024_final.groupby(['Zip Code', 'Restaurant Type']).size().reset_index(name='Count')
zip_totals = df_2024_final.groupby('Zip Code').size().reset_index(name='Total_in_Zip')

# 2. Identify Top 5 for each category
top_local_zips = type_counts[type_counts['Restaurant Type'] == 'Local'].nlargest(5, 'Count')['Zip Code']
top_chain_zips = type_counts[type_counts['Restaurant Type'] == 'Chain'].nlargest(5, 'Count')['Zip Code']

# Combine them into a single list of unique zip codes
selected_zips = pd.concat([top_local_zips, top_chain_zips]).unique()

# 3. Filter the plotting dataframe
plot_df_focused = pd.merge(type_counts, zip_totals, on='Zip Code')
plot_df_focused = plot_df_focused[plot_df_focused['Zip Code'].isin(selected_zips)]

# Recalculate percentages for the focused set
plot_df_focused['Composition_Pct'] = (plot_df_focused['Count'] / plot_df_focused['Total_in_Zip']) * 100

# 4. Create the Plot
fig5_narrow = px.bar(
    plot_df_focused,
    x='Zip Code',
    y='Count',
    color='Restaurant Type',
    title='Top Austin Restaurant Hubs: Local vs. Chain (2024)',
    hover_data={
        'Zip Code': True,
        'Restaurant Type': True,
        'Count': True,
        'Composition_Pct': ':.2f%',
        'Total_in_Zip': True
    },
    labels={
        'Count': 'Establishment Count',
        'Composition_Pct': 'Composition %',
        'Total_in_Zip': 'Total in Zip'
    },
    barmode='stack',
    color_discrete_map={'Local': '#1B4332', 'Chain': '#00B4D8'}
)


fig5_narrow.update_layout(xaxis={'categoryorder':'total descending'})

fig5_narrow.show()

Argument 6 - Looking at how local and chain restaurants compare in inspection scores

In [ ]:
fig6 = px.box(df_2024_final,
             x='Restaurant Type',
             y='Score',
             color='Restaurant Type',
             points="outliers",
             color_discrete_map={'Local': '#1B4332', 'Chain': '#00B4D8'},
             title='Inspection Score Statistics: Local vs. Chain',
             notched=False)

fig6.update_traces(boxmean=True)

fig6.update_layout(
    xaxis_title="Restaurant Category",
    yaxis_title="Inspection Score",
    showlegend=False
)

fig6.show()

In [ ]:
# SCATTER PLOT: Income vs. Inspection Scores (Local vs. Chain)
# Aggregate scores by Zip Code and Restaurant Type
scatter_df = df_2024_final.groupby(['Zip Code', 'Restaurant Type']).agg({
    'Score': 'mean',
    'Median_Income': 'first'
}).reset_index()

fig_scatter = px.scatter(
    scatter_df,
    x='Median_Income',
    y='Score',
    color='Restaurant Type',
    trendline="ols",
    title='Impact of Wealth on Scores: Local vs. Chain Restaurants',
    labels={'Median_Income': 'Median Income ($)', 'Score': 'Avg Inspection Score'},
    hover_data=['Zip Code'],
    color_discrete_map={'Local': '#1B4332', 'Chain': '#00B4D8'}
)
fig_scatter.update_layout(xaxis_tickformat='$,.0f')
fig_scatter.show()

# 3. Optional: See the specific correlation coefficients
for r_type in ['Local', 'Chain']:
    subset = type_income_stats[type_income_stats['Restaurant Type'] == r_type].dropna()
    corr, _ = pearsonr(subset['Median_Income'], subset['Score'])
    print(f"Correlation for {r_type} restaurants: {corr:.3f}")

Correlation for Local restaurants: 0.354
Correlation for Chain restaurants: -0.075


In [ ]:
# 1. Prepare the data (ensure df_final is your merged dataframe)
# Using 'df_final' from your previous notebook steps
income_col = 'Median_Income'

target_zips = ['78701', '78660', '78753', '78758', '78745', '78705', '78741']

# Aggregate by Zip Code
zip_stats = df_final.groupby('Zip Code').agg({
    'Median_Income': 'first',
    'Restaurant Name': 'count',
    'Restaurant Type': lambda x: (x == 'Local').sum() / len(x) * 100
}).reset_index()

zip_stats.columns = ['Zip Code', 'Median Income', 'Total Establishments', 'Percent Local']

# List of Zip Codes you want to show
target_zips = ['78701', '78660', '78753', '78758', '78745', '78705', '78741']

# Create a column that is empty unless the Zip Code is in your list
zip_stats['Label'] = zip_stats['Zip Code'].apply(lambda x: x if x in target_zips else '')

# Create the Bubble Chart with Selective Labels
fig_selective = px.scatter(
    zip_stats,
    x='Median Income',
    y='Percent Local',
    size='Total Establishments',
    color='Percent Local',
    hover_name='Zip Code',
    text='Label',  # Use the custom column for the labels
    color_continuous_scale='Viridis',
    title='Austin Neighborhoods: Focus on Major Hubs',
    labels={'Percent Local': '% of Restaurants that are Local'}
)

# Improve readability
fig_selective.update_traces(textposition='top center')
fig_selective.update_layout(
    xaxis_tickformat='$,.0f',
    yaxis_ticksuffix='%',
    xaxis_title="Neighborhood Median Income",
    yaxis_title="Percent of Local Establishments"
)

fig_selective.show()

# # 2. Create the Bubble Chart
# fig_hover = px.scatter(
#     zip_stats,
#     x='Median Income',
#     y='Percent Local',
#     size='Total Establishments',
#     color='Percent Local',
#     hover_name='Zip Code',  # This keeps the Zip Code visible on hover
#     color_continuous_scale='Viridis',
#     title='Austin Neighborhoods: Wealth vs. Local Business Density',
#     labels={'Percent Local': '% of Restaurants that are Local'}
# )

# fig_hover.update_layout(
#     xaxis_tickformat='$,.0f',
#     yaxis_ticksuffix='%',
#     xaxis_title="Neighborhood Median Income",
#     yaxis_title="Percent of Local Establishments"
# )

# fig_hover.show()

Final Conclusion

In [ ]:
# --- Step 1: filter your real dataset ---
df_2024_final['Zip Code'] = df_2024_final['Zip Code'].astype(str)

zip_78705 = df_2024_final[df_2024_final['Zip Code'] == '78705'].copy()

# --- Step 2: get coordinates using pgeocode ---
nomi = pgeocode.Nominatim("us")

zip_78705["Latitude"] = zip_78705["Zip Code"].apply(lambda x: nomi.query_postal_code(x).latitude)
zip_78705["Longitude"] = zip_78705["Zip Code"].apply(lambda x: nomi.query_postal_code(x).longitude)

# --- Step 3: drop missing ---
zip_78705 = zip_78705.dropna(subset=["Latitude", "Longitude"])

# --- Step 4: jitter (so points don't overlap) ---
zip_78705["Latitude"] += np.random.normal(0, 0.002, len(zip_78705))
zip_78705["Longitude"] += np.random.normal(0, 0.002, len(zip_78705))

print("Rows:", len(zip_78705))

# --- Step 5: map ---
map_78705 = alt.Chart(zip_78705).mark_circle(
    size=80,
    opacity=0.7
).encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    color=alt.Color('Score:Q', scale=alt.Scale(scheme='redyellowgreen')),
    tooltip=[
        alt.Tooltip('Restaurant Name:N', title='Restaurant'),
        alt.Tooltip('Score:Q', title='Score'),
        alt.Tooltip('Restaurant Type:N', title='Type')
    ]
).properties(
    width=500,
    height=400,
    title='Restaurants in ZIP Code 78705'
)

map_78705

Rows: 282


alt.Chart(...)